# Lab 7.06 - Time series analysis

In [ ]:
# Package importsLab7.06

covid=pd.read_csv('https://raw.githubusercontent.com/HoGentTIN/dsai-labs/refs/heads/main/data/SP500.csv',delimiter=',',parse_dates=['Date'],index_col=['Date'])
covid.head()

covid_be= covid19[covid19['location']=='Belgium'][['new_cases']]
covid_be.head()

covid_be.plot(y='new_cases',figsize=[10,5])

covid_be['SMA7'] = covid_be['new_cases'].rolling(7).mean()
covid_be['SMA30'] = covid_be['new_cases'].rolling(30).mean()
covid_be.plot(y=['new_cases','SMA7','SMA30'],figsize=[10,5])

Holt-Winters model for new cases early 2021

deel_covid_be = covid_be['2020-12-01':'2021-02-28']
deel_covid_be.plot(y=['new_cases','SMA7','SMA30'],figsize=[10,5])

forecast for 21 days
from statsmodels.tsa.holtwinters import ExponentialSmoothing

data_tes = ExponentialSmoothing(deel_covid_be['new_cases'], trend='add', seasonal='add', seasonal_periods=7, freq='D').fit()
data_tes_fcast= data_tes.forecast(21)

covid_be['2021-03-01':'2021-03-21'].plot(legend=True)
data_tes_fcast.plot(legend=True)


•	Calculate the Mean Squared Error for a forecast period of 7 days, and compare its square root with the standard deviation of observed new cases over the test period and forecasted period combined.
•	Do the same for a forecast period of 14 and 21 days.
•	For which period is the quality of the forecast ok?

from sklearn.metrics import mean_squared_error, mean_absolute_error

#mse7

y_true = covid_be['2021-03-01':'2021-03-07']['new_cases'].values     #effectieve waarden
y_predicted = data_tes_fcast[:7].values				#voorspelde waarden
print(f'mse7: {mean_squared_error(y_true, y_predicted)}')




mse14
y_true = covid_be['2021-03-01':'2021-03-14']['new_cases'].values
y_predicted = data_tes_fcast[:14].values
print(f'mse14: {mean_squared_error(y_true, y_predicted)}')

mse21
y_true = covid_be['2021-03-01':'2021-03-21']['new_cases'].values
y_predicted = data_tes_fcast[:21].values
print(f'mse21: {mean_squared_error(y_true, y_predicted)}')

icu_be = covid19[covid19['location']=='Belgium'][['icu_patients']]
icu_be = icu_be.dropna()
icu_be.head()
icu_be.plot(y='icu_patients',figsize=[10,5])

from statsmodels.tsa.api import Holt

data_des = Holt(icu_be['icu_patients']).fit()
data_des_fcast= data_des.forecast(70)

data_des.trend[-1]
data_des.level[-1]


alpha en Beta
data_des.params_formatted
	name	param	optimized
smoothing_level	alpha	0.958466	True
smoothing_trend	beta	0.373023	True
initial_level	l.0	53.000000	False
initial_trend	b.0	26.000000	False


from statsmodels.tsa.api import Holt

data_des = Holt(icu_be['icu_patients']).fit(smoothing_level=.1,smoothing_trend=.1)
data_des_fcast= data_des.forecast(70)
icu_be['icu_patients'].plot(legend=True)
data_des_fcast.plot(marker='.',legend=True, label='Forecast DES')

Lab  7.07

data = pd.read_csv('https://raw.githubusercontent.com/HoGentTIN/dsai-labs/refs/heads/main/data/SP500.csv', delimiter = ",", parse_dates=['Date']).set_index(['Date'])
data.head()
dubbele haakjes gebruiken om de kolom aan te duiden
data=data[['Close']]
data.head()
data.plot(y='Close', figsize=[20,10])
data['Close'] = data['Close'].astype(float)
data['SMA50'] = data['Close'].rolling(50).mean()
data['SMA200'] = data['Close'].rolling(200).mean()
data.plot(y=['Close', 'SMA50', 'SMA200'], figsize=[20,10])




## Exercise 6: COVID-19 data


In this lab assignment, we will make use of the COVID-19 dataset maintained by [Our World in Data](https://ourworldindata.org/coronavirus), published on Github at <https://github.com/owid/covid-19-data/tree/master/public/data>.

First, we import the dataset, parse the `date` column as the Python `DateTime` type and set this as the index:

In [ ]:
covid19 = pd.read_csv('https://covid.ourworldindata.org/data/owid-covid-data.csv', parse_dates=['date']).set_index(['date'])
covid19.head()

Create a new `DataFrame` (with name e.g. `covid19_be`) that only contains the new cases in Belgium and use the `plot()` method of `DataFrame` to visualize it. Increase the size of the picture with the `figsize` parameter, otherwise it will be too small.

### Moving average

Add new columns to the data frame with new cases in Belgium with the simple moving average for 7 and 30 days. Plot the entire data frame (observations and both moving averages).

### Holt-Winters model for new cases early 2021

The period from about December 2020 up to the end of February 2021 seems quite regular. Create a new `DataFrame` and select only the observations during that period (1 December 2020 to 28 February 2021). Plot the `DataFrame` (it will still contain the moving averages from the previous step).



Build a Holt-Winters model for the observed new cases during that period.

Use the additive type for both trend and seasonal smoothing. Set the value for `seasonal_periods` to the appropriate value! Plot the observed and fitted values.

Now, make a forecast for 21 days and plot observed and forecasted values. What do you notice when you compare observed and forecasted values as time progresses?

### Evaluating model quality


- Calculate the Mean Squared Error for a forecast period of 7 days, and compare its square root with the standard deviation of observed new cases over the test period and forecasted period combined.
- Do the same for a forecast period of 14 and 21 days.
- For which period is the quality of the forecast ok?

The expected results are shown in the table below:

|     Forecast period      |         MSE |     √MSE |    stdev |
| :----------------------: | ----------: | -------: | -------: |
| 2021-03-01 to 2021-03-07 |   25408.902 |  159.402 |  855.684 |
| 2021-03-01 to 2021-03-14 |  154280.817 |  392.786 |  895.531 |
| 2021-03-01 to 2021-03-21 | 1048835.781 | 1024.127 | 1052.978 |

### ICU patients

Create a new `DataFrame` with only the total number of ICU (intensive care units) patients in the Belgian hospitals. Make sure that all NaN's are removed, since time series analysis functions can't cope with missing values. Plot this time series.

Build a duible exponential smoothing (Holt) model of this time series. Make a forecast of 70 time units and plot the observations, fitted values and forecast.

Are we currently in an upward or downward trend, according to this model? Do the forecasted values seem reasonable if you look at the last period?

Since we didn't set any initial parameters like $\alpha$ and $\beta$, the model calculated them using some rule of thumb. List these parameters from the model:

What were the final estimated values for the level and trend, that are used in the forecast? In other words, what are the parameters of the line that estimates future observations?

Let's try to set the parameters ourselves. Recreate the Holt-model with $\alpha = \beta = 0.1$. Calculate a forecast like before and plot.

Compare the model parameters and final estimates. Which of the two forecast models seems to perform best if you look at the plots?